In [ ]:
import time

notebook_start_time = time.time()
print(f"Notebook execution started at: {time.ctime(notebook_start_time)}")

Notebook execution started at: Fri Oct  2 22:57:24 2026


## Submission summary

| | |
|---|---|
| **Group number** | **29** |
| **Your full name** | **Onuche Victor Oma-ojo** |
| **Dataset** | BankChurners (Credit Card Customers, Kaggle `sakshigoyal7/credit-card-customers`): 10,127 customers, 16.07% churn |
| **Depth track** | Classification Depth |
| **Team members** | 1. Onuche Victor Oma-ojo
2. ⁠Abiye Gloria Genesis
3. ⁠Godwin Charity Chinenye
4. Gabriel  Irahyhohe  ifeoluwa
5. ⁠Itunuoluwa  Ayilara |
| **Final model** | Gradient Boosting (`GradientBoostingClassifier`: 100 trees, `max_depth=3`, `random_state=42`), no class weights, threshold 0.12 |
| **Stratified dummy F1 / ROC AUC** | 0.17 / 0.51 |
| **Final model F1 / ROC AUC** | 0.81 / 0.99 |
| **Leakage columns dropped** | `Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1`<br>`Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2` |
| **Full run time on Colab** | [fill in after Restart & Run All on Colab] minutes |

F1 and ROC AUC are measured on the held-out test set (2,026 customers, stratified 80/20 split, `random_state=42`). The stratified dummy is `DummyClassifier(strategy="stratified")`, which guesses at random in proportion to the class rates. The final model uses threshold 0.12, chosen on out-of-fold training predictions to minimise an assumed cost (missed churner \$300, false alarm \$30). That is why its F1 (0.81) is lower than at the default 0.5 threshold (0.87): at 0.12 it catches 95% of churners at 0.71 precision. Repeating the threshold search with 10 different cross-validation shuffles gave values between 0.09 and 0.15, so 0.12 sits in a flat, low-cost region rather than being a single exact optimum; on the test set it cost \$8,310 against \$7,800 at the best point (0.13). The two leakage columns alone predict churn with 100% accuracy (Part 1.5). `CLIENTNUM` (an ID) and `Avg_Open_To_Buy` (exactly `Credit_Limit` minus `Total_Revolving_Bal`) were also dropped, but they are not leakage.

# TS Academy Capstone — Credit Card Customer Churn
**Author:** Onuche Victor  |  **Programme:** TS Academy, Data Science and Machine Learning
**Dataset:** BankChurners (Credit Card Customers, Kaggle — `sakshigoyal7/credit-card-customers`)

**Business question:** which credit-card customers are about to leave the bank, and what do they look like?

## Contents

| Part | Topic |
|---|---|
| **1** | Data loading, SQL queries, EDA and cleaning |
| **2** | Feature engineering, encoding and scaling |
| **3** | Clustering analysis (K-Means, PCA) |
| **4** | Classification |
| **5** | Imbalance handling |
| **6** | The tie-in: clusters versus churn |
| **7** | Conclusion and recommendations |

## 0. Setup

All imports live in this one cell: `pandas`/`numpy` (tables and maths), `sqlite3` (the database, built into
Python), `matplotlib`/`seaborn` (plots), `scikit-learn` (scaling, clustering, models, metrics), `scipy`
(one statistical test) and `imbalanced-learn` (SMOTE).

In [ ]:
import os
import sqlite3
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             silhouette_score, confusion_matrix, precision_recall_curve,
                             ConfusionMatrixDisplay, RocCurveDisplay)

try:
    from imblearn.over_sampling import SMOTE                    # used in Part 5
    from imblearn.pipeline import Pipeline as ImbPipeline
except ImportError:
    raise ImportError("SMOTE needs the imbalanced-learn package. Run:  pip install imbalanced-learn")

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)   # so wide tables are not truncated
sns.set_style("whitegrid")
print("Libraries loaded")

Libraries loaded


---
# Part 1 — Data Loading, SQL, EDA and Cleaning

## 1.1 Load the dataset from Kaggle

`kagglehub` downloads the dataset straight from Kaggle into the notebook (this needs
`pip install kagglehub` and a Kaggle API token). If that is not available, the cell falls back to a local
copy of `BankChurners.csv`, looking in the notebook folder, then Downloads, then Desktop. The file is
identical either way, so the rest of the notebook does not change.

In [ ]:
csv_path = None
import kagglehub
kaggle_folder = kagglehub.dataset_download("sakshigoyal7/credit-card-customers")
csv_path = os.path.join(kaggle_folder, "BankChurners.csv")
print("Downloaded from Kaggle:", csv_path)

df = pd.read_csv(csv_path)
print("Loaded from:", csv_path)
print("Rows, columns:", df.shape)
df.head()

Using Colab cache for faster access to the 'credit-card-customers' dataset.
Downloaded from Kaggle: /kaggle/input/credit-card-customers/BankChurners.csv
Loaded from: /kaggle/input/credit-card-customers/BankChurners.csv
Rows, columns: (10127, 23)


,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998
3,769911858,Existing Customer,40,F,4,High School,Unknown,Less than $40K,Blue,34,3,4,1,3313.0,2517,796.0,1.405,1171,20,2.333,0.760,0.000134,0.99987
4,709106358,Existing Customer,40,M,3,Uneducated,Married,$60K - $80K,Blue,21,5,1,0,4716.0,0,4716.0,2.175,816,28,2.500,0.000,0.000022,0.99998


## 1.2 Push the data into a SQLite database

`to_sql` writes the whole DataFrame into a table called `customers` inside a database file
(`bank_churn.db`). From now on we can query it with real SQL, which is how this data would live in
a bank's system.

In [ ]:
conn = sqlite3.connect("bank_churn.db")          # creates the .db file if it does not exist
df.to_sql("customers", conn, if_exists="replace", index=False)

# confirm it landed
pd.read_sql("SELECT COUNT(*) AS rows_in_db FROM customers;", conn)

,rows_in_db
0,10127


## 1.3 SQL queries that tell us something useful

Four queries, each followed by what it taught us.

### Query 1 — how many customers actually churn?

In [ ]:
q1 = pd.read_sql('''
SELECT Attrition_Flag,
       COUNT(*) AS customers,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM customers), 2) AS percent
FROM customers
GROUP BY Attrition_Flag;
''', conn)
q1

,Attrition_Flag,customers,percent
0,Attrited Customer,1627,16.07
1,Existing Customer,8500,83.93


**What I learned:** only **16.07%** of the 10,127 customers churned (1,627 of them); 83.93% are still
active. The target is **imbalanced roughly 1:5**. This single number shapes the whole project: a lazy
model that predicts "nobody churns" would already be 84% accurate, so accuracy alone will be a
misleading metric later, and Part 5 (imbalance handling) is genuinely needed.

### Query 2 — does churn differ by income bracket?

In [ ]:
q2 = pd.read_sql('''
SELECT Income_Category,
       COUNT(*) AS customers,
       ROUND(100.0 * SUM(CASE WHEN Attrition_Flag = 'Attrited Customer' THEN 1 ELSE 0 END)
             / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY Income_Category
ORDER BY churn_rate_pct DESC;
''', conn)
q2

,Income_Category,customers,churn_rate_pct
0,$120K +,727,17.33
1,Less than $40K,3561,17.19
2,Unknown,1112,16.82
3,$80K - $120K,1535,15.77
4,$40K - $60K,1790,15.14
5,$60K - $80K,1402,13.48


**What I learned:** churn is **U-shaped across income**: the highest earners (\$120K+, 17.3%) and the
lowest earners (<\$40K, 17.2%) leave most, while the \$60K–\$80K middle is the most loyal (13.5%).
The gap is real but modest (~4 points), so income will probably be a weak predictor on its own.
I also see that **1,112 customers have `Income_Category = 'Unknown'`** — a category, not an empty
cell, which matters for the cleaning step below.

### Query 3 — does holding more products keep customers?

In [ ]:
q3 = pd.read_sql('''
SELECT Total_Relationship_Count AS products_held,
       COUNT(*) AS customers,
       ROUND(100.0 * SUM(CASE WHEN Attrition_Flag = 'Attrited Customer' THEN 1 ELSE 0 END)
             / COUNT(*), 2) AS churn_rate_pct
FROM customers
GROUP BY products_held
ORDER BY products_held;
''', conn)
q3

,products_held,customers,churn_rate_pct
0,1,910,25.60
1,2,1243,27.84
2,3,2305,17.35
3,4,1912,11.77
4,5,1891,12.00
5,6,1866,10.50


**What I learned:** this is the strongest signal so far. Customers holding only **1–2 products churn at
25.6% and 27.8%**, against **10.5% for those holding 6**. Each extra product the customer holds with the
bank roughly halves the risk by the time you reach 3+. Business reading: cross-selling is retention.
`Total_Relationship_Count` should be a powerful feature, and it is a good candidate for the
interaction feature I build in Part 2.

### Query 4 — how do leavers behave compared to stayers?

In [ ]:
q4 = pd.read_sql('''
SELECT Attrition_Flag,
       ROUND(AVG(Total_Trans_Ct), 1)        AS avg_transactions,
       ROUND(AVG(Total_Trans_Amt), 0)       AS avg_spend,
       ROUND(AVG(Avg_Utilization_Ratio), 3) AS avg_utilisation,
       ROUND(AVG(Months_Inactive_12_mon), 2) AS avg_months_inactive
FROM customers
GROUP BY Attrition_Flag;
''', conn)
q4

,Attrition_Flag,avg_transactions,avg_spend,avg_utilisation,avg_months_inactive
0,Attrited Customer,44.9,3095.0,0.162,2.69
1,Existing Customer,68.7,4655.0,0.296,2.27


**What I learned:** churners are **quiet customers before they leave**. They make 44.9 transactions a
year versus 68.7 for stayers, spend \$3,095 versus \$4,655, use only 16.2% of their credit limit versus
29.6%, and sit idle slightly longer (2.69 vs 2.27 inactive months). So churn here is mostly a
**disengagement story, not a demographics story** — transaction behaviour is where the predictive power
will come from, which is exactly why my engineered features in Part 2 focus on activity intensity.

## 1.4 Exploratory Data Analysis

### Shape, data types and missing values

In [ ]:
print("Shape:", df.shape, "\n")
print(df.dtypes, "\n")
print("Missing values per column:")
print(df.isnull().sum().sum(), "missing values in total")

Shape: (10127, 23) 

CLIENTNUM                                                                                                                               int64
Attrition_Flag                                                                                                                         object
Customer_Age                                                                                                                            int64
Gender                                                                                                                                 object
Dependent_count                                                                                                                         int64
Education_Level                                                                                                                        object
Marital_Status                                                                                                                 

**What I learned:** 10,127 rows and 23 columns; 6 text columns and 17 numeric ones, and
**zero missing values** as far as pandas is concerned. That is misleading — the missingness in this
dataset is hidden inside the text columns as the literal word `"Unknown"`, which I check next.

In [ ]:
# hidden missingness: 'Unknown' used as a placeholder
text_cols = df.select_dtypes(include="object").columns
for col in text_cols:
    share = (df[col] == "Unknown").mean()
    if share > 0:
        print(f"{col}: {share:.1%} 'Unknown'")

# structural duplicates
print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate customer IDs:", df['CLIENTNUM'].duplicated().sum())

Education_Level: 15.0% 'Unknown'
Marital_Status: 7.4% 'Unknown'
Income_Category: 11.0% 'Unknown'

Duplicate rows: 0
Duplicate customer IDs: 0


**What I learned:** `Education_Level` is 15.0% Unknown, `Income_Category` 11.0%, `Marital_Status` 7.4%.
There are **no duplicate rows and no repeated `CLIENTNUM`**, so every row is one distinct customer and
nothing needs to be de-duplicated.

### Target distribution

In [ ]:
ax = sns.countplot(data=df, x="Attrition_Flag", hue="Attrition_Flag",
                   palette="Set2", legend=False)
ax.set_title("Target distribution: Attrition_Flag")
ax.set_xlabel("")
for p in ax.patches:                        # print the % on top of each bar
    pct = 100 * p.get_height() / len(df)
    ax.annotate(f"{pct:.1f}%", (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.show()

<Figure size 640x480 with 1 Axes>

**What I learned:** the same 84/16 split the SQL showed, now visible. Minority class = churners = the
class we actually care about. Every model in Part 4 must be judged on **recall and F1 for that
minority class**, not on accuracy.

### Feature distributions (five features)

I plot five features chosen to cover one demographic, three behavioural and one credit feature, and
split each by churn so the plot is diagnostic rather than decorative.

In [ ]:
features = ["Customer_Age", "Total_Trans_Ct", "Total_Trans_Amt",
            "Avg_Utilization_Ratio", "Credit_Limit"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flatten(), features):
    sns.histplot(data=df, x=col, hue="Attrition_Flag", bins=30,
                 kde=False, palette="Set2", ax=ax)
    ax.set_title(col)
axes.flatten()[-1].axis("off")              # blank the 6th, unused panel
plt.tight_layout()
plt.show()

df[features].describe()

<Figure size 1600x800 with 6 Axes>

,Customer_Age,Total_Trans_Ct,Total_Trans_Amt,Avg_Utilization_Ratio,Credit_Limit
count,10127.000000,10127.000000,10127.000000,10127.000000,10127.000000
mean,46.325960,64.858695,4404.086304,0.274894,8631.953698
std,8.016814,23.472570,3397.129254,0.275691,9088.776650
min,26.000000,10.000000,510.000000,0.000000,1438.300000
25%,41.000000,45.000000,2155.500000,0.023000,2555.000000
50%,46.000000,67.000000,3899.000000,0.176000,4549.000000
75%,52.000000,81.000000,4741.000000,0.503000,11067.500000
max,73.000000,139.000000,18484.000000,0.999000,34516.000000


**What I learned, feature by feature:**

- **`Customer_Age`** — roughly normal, 26 to 73, centred near 46. Churners and stayers overlap almost
  completely, so age alone separates nothing.
- **`Total_Trans_Ct`** — the clearest separator in the dataset. Churners pile up below ~45
  transactions; stayers dominate the 60–90 range. This is bimodal overall, which is itself a hint
  that there are natural customer segments — good news for the K-Means work in Part 3.
- **`Total_Trans_Amt`** — right-skewed with a long tail of heavy spenders, and churners sit in the
  low-spend hump.
- **`Avg_Utilization_Ratio`** — heavily right-skewed and stacked at 0: a very large group never
  revolves a balance. Churners are concentrated in that zero spike.
- **`Credit_Limit`** — strongly right-skewed with a spike at the \$34,516 cap. Similar for both
  classes, so limit is more a bank decision than a churn signal.

**Consequence for modelling:** several features are skewed and on wildly different numeric ranges
(utilisation 0–1 vs credit limit up to 34,516), so **scaling is not optional** for the
distance-based methods (K-Means, PCA, KNN, logistic regression).

## 1.5 Cleaning — every step justified

**No blanket `fillna`.** There is nothing to fill: pandas reports zero nulls. The only cleaning
decisions are about *columns that should not be in the model* and about *how to treat "Unknown"*.

In [ ]:
clean = df.copy()

# --- Step 1: drop the two Naive Bayes columns -------------------------------
leaky = [c for c in clean.columns if c.startswith("Naive_Bayes_Classifier")]
print("Dropping leakage columns:", len(leaky))
clean = clean.drop(columns=leaky)

# --- Step 2: drop the ID column --------------------------------------------
clean = clean.drop(columns=["CLIENTNUM"])

# --- Step 3: drop one redundant column -------------------------------------
gap = (df["Credit_Limit"] - df["Total_Revolving_Bal"] - df["Avg_Open_To_Buy"]).abs().max()
print("Max gap in Credit_Limit - Total_Revolving_Bal - Avg_Open_To_Buy:", round(gap, 6))
clean = clean.drop(columns=["Avg_Open_To_Buy"])

# --- Step 4: turn the target into 0/1 --------------------------------------
clean["Churn"] = (clean["Attrition_Flag"] == "Attrited Customer").astype(int)
clean = clean.drop(columns=["Attrition_Flag"])

print("\nShape after cleaning:", clean.shape)
print("Churn rate:", round(clean['Churn'].mean(), 4))
display(clean.head(3))
print(clean.shape)

Dropping leakage columns: 2
Max gap in Credit_Limit - Total_Revolving_Bal - Avg_Open_To_Buy: 0.0

Shape after cleaning: (10127, 19)
Churn rate: 0.1607


,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Churn
0,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,12691.0,777,1.335,1144,42,1.625,0.061,0
1,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,1,2,8256.0,864,1.541,1291,33,3.714,0.105,0
2,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,1,0,3418.0,0,2.594,1887,20,2.333,0.000,0


(10127, 19)


**Justification for each step:**

1. **Dropped the two `Naive_Bayes_Classifier_...` columns.** These are not customer attributes at all —
   they are the output probabilities of somebody else's model that was trained on the target. Keeping
   them is **target leakage**: my classifiers would score ~99% by copying that model's answer and
   would be useless on new customers. This is the single most important cleaning decision in the
   project.
2. **Dropped `CLIENTNUM`.** An account identifier. It is numeric, so a model would happily treat it as
   a magnitude, but the number carries no behavioural meaning and cannot generalise to new IDs.
3. **Dropped `Avg_Open_To_Buy`.** The check above shows it equals
   `Credit_Limit − Total_Revolving_Bal` to within floating-point error (max gap ≈ 5.7e-14), i.e. it is
   perfectly collinear with two columns I am keeping. Redundant columns inflate multicollinearity and
   distort distance-based clustering by counting the same information twice. I keep `Credit_Limit` and
   `Total_Revolving_Bal` because they are the raw, interpretable quantities.
4. **Encoded the target as `Churn` (1 = Attrited, 0 = Existing).** scikit-learn needs a numeric target,
   and making the **churner the positive class (1)** means `precision`, `recall` and `ROC AUC` are
   reported for the class the bank actually wants to catch.

**Rows dropped: none.** There were no duplicates, no impossible values (ages 26–73, no negative
balances, no ratios outside 0–1), so deleting rows would only throw away information.

**On `"Unknown"`:** I deliberately **keep it as its own category** rather than imputing it. Two reasons.
First, it covers 7–15% of three columns — imputing with the mode would invent data for ~1,500
customers. Second, refusing to disclose income or education can itself be informative, and the SQL in
Query 2 showed Unknown-income customers churn at 16.8%, i.e. their own distinct level rather than the
average. Encoding Unknown as a category lets the model decide whether it matters; imputing would
destroy that signal.

---
# Part 2 — Feature Engineering and Encoding

## 2.1 Three new features (plus a bin and an interaction)

The EDA said churn is a **disengagement** story, so the engineered features all sharpen the measurement
of engagement rather than adding new demographics.

In [ ]:
# Feature 1 — RATIO: average value of a single transaction
clean["Avg_Trans_Value"] = clean["Total_Trans_Amt"] / clean["Total_Trans_Ct"]

# Feature 2 — RATIO normalised by tenure: how often does this customer transact per month?
clean["Trans_Per_Month"] = clean["Total_Trans_Ct"] / clean["Months_on_book"]

# Feature 3 — RATIO: service contacts per product held (complaint intensity)
clean["Contacts_Per_Product"] = clean["Contacts_Count_12_mon"] / clean["Total_Relationship_Count"]

# Feature 4 — BINNING: age brackets instead of raw age
clean["Age_Group"] = pd.cut(clean["Customer_Age"],
                            bins=[25, 35, 45, 55, 65, 100],
                            labels=["26-35", "36-45", "46-55", "56-65", "65+"])

# Feature 5 — INTERACTION: inactivity x number of service contacts
clean["Inactive_x_Contacts"] = clean["Months_Inactive_12_mon"] * clean["Contacts_Count_12_mon"]

new_cols = ["Avg_Trans_Value", "Trans_Per_Month", "Contacts_Per_Product",
            "Age_Group", "Inactive_x_Contacts"]
clean[new_cols].head()

,Avg_Trans_Value,Trans_Per_Month,Contacts_Per_Product,Age_Group,Inactive_x_Contacts
0,27.238095,1.076923,0.600000,36-45,3
1,39.121212,0.750000,0.333333,46-55,2
2,94.350000,0.555556,0.000000,46-55,0
3,58.550000,0.588235,0.333333,36-45,4
4,29.142857,1.333333,0.000000,36-45,0


In [ ]:
# sanity check: do the new features separate the classes?
for col in ["Avg_Trans_Value", "Trans_Per_Month", "Contacts_Per_Product", "Inactive_x_Contacts"]:
    print(col, "-> mean by churn:")
    print(clean.groupby("Churn")[col].mean().round(3).to_string(), "\n")

print("Churn rate by Age_Group:")
print((100 * clean.groupby("Age_Group", observed=True)["Churn"].mean()).round(1).to_string())
print("\nAge_Group counts (no empty bins, no NaN):")
print(clean["Age_Group"].value_counts(dropna=False).to_string())

Avg_Trans_Value -> mean by churn:
Churn
0    62.425
1    63.593 

Trans_Per_Month -> mean by churn:
Churn
0    2.038
1    1.324 

Contacts_Per_Product -> mean by churn:
Churn
0    0.746
1    1.230 

Inactive_x_Contacts -> mean by churn:
Churn
0    5.353
1    8.017 

Churn rate by Age_Group:
Age_Group
26-35    13.3
36-45    16.2
46-55    16.6
56-65    15.8
65+      20.0

Age_Group counts (no empty bins, no NaN):
Age_Group
46-55    4135
36-45    3742
56-65    1321
26-35     919
65+        10


**Business reasoning, feature by feature:**

1. **`Avg_Trans_Value` = `Total_Trans_Amt / Total_Trans_Ct`.** Total spend confounds two different
   customers: someone who makes 100 small purchases and someone who makes 5 large ones can look
   identical. The ratio separates **basket size from frequency**, which matters commercially because a
   card used for many small everyday purchases is embedded in the customer's routine (sticky), while a
   card used occasionally for big-ticket items is easy to replace. In this dataset the average ticket turns out to be almost
   identical for churners and stayers (\$63.6 vs \$62.4), which is itself a finding: the gap in total
   spend is driven **entirely by frequency, not by basket size**. That is useful — it tells the bank the
   lever is usage occasions, not big-ticket purchases.
2. **`Trans_Per_Month` = `Total_Trans_Ct / Months_on_book`.** Raw transaction count penalises new
   customers, who have had less time to accumulate transactions. Dividing by tenure turns it into an
   **intensity rate** that is comparable across a 13-month customer and a 56-month customer. This is
   the fair version of "how active is this person".
3. **`Contacts_Per_Product` = `Contacts_Count_12_mon / Total_Relationship_Count`.** Six calls from a
   customer holding six products is routine servicing; six calls from a single-product customer is
   friction. Normalising contacts by products held converts a raw call count into a
   **dissatisfaction-per-relationship** measure. Churners average **1.23 contacts per product against 0.75** for stayers — the largest
   relative gap among my engineered features.
4. **`Age_Group` (binned `Customer_Age`).** The age histogram showed no linear relationship with churn,
   and the bank's retention offers are designed for life-stage segments, not single years. Binning
   lets a model capture a **non-monotonic** age effect and makes the output speak the language the
   marketing team uses. The bin edges follow standard 10-year brackets; the smallest bin (65+, 10
   customers) is retained because dropping it would drop real customers, and it is a single ordinal
   level rather than a sparse dummy.
5. **`Inactive_x_Contacts` (interaction).** Inactivity is ambiguous on its own — a quiet customer may
   be content or may be gone. Combined with service contacts, the meaning sharpens: **idle *and*
   calling** is a customer with an unresolved problem and no reason to stay. The product is high only
   when both are high, which is precisely the at-risk combination, and it lets linear models represent
   a relationship they otherwise could not. Churners score **8.02 on average against 5.35** for stayers.

## 2.2 Encoding every categorical column

There are five text/categorical columns left. I choose the encoder per column rather than applying one
blindly.

In [ ]:
print(clean.select_dtypes(include=["object", "category"]).nunique(), "\n")
for col in ["Gender", "Card_Category", "Education_Level", "Marital_Status", "Income_Category"]:
    print(col, "->", list(clean[col].unique()))

Gender             2
Education_Level    7
Marital_Status     4
Income_Category    6
Card_Category      4
Age_Group          5
dtype: int64 

Gender -> ['M', 'F']
Card_Category -> ['Blue', 'Gold', 'Silver', 'Platinum']
Education_Level -> ['High School', 'Graduate', 'Uneducated', 'Unknown', 'College', 'Post-Graduate', 'Doctorate']
Marital_Status -> ['Married', 'Single', 'Unknown', 'Divorced']
Income_Category -> ['$60K - $80K', 'Less than $40K', '$80K - $120K', '$40K - $60K', '$120K +', 'Unknown']


In [ ]:
encoded = clean.copy()

# --- (a) LABEL encoding: binary column ---
encoded["Gender"] = encoded["Gender"].map({"F": 0, "M": 1})

# --- (b) ORDINAL label encoding: genuinely ranked, no 'Unknown' ---
encoded["Card_Category"] = encoded["Card_Category"].map(
    {"Blue": 0, "Silver": 1, "Gold": 2, "Platinum": 3})

encoded["Age_Group"] = encoded["Age_Group"].map(
    {"26-35": 0, "36-45": 1, "46-55": 2, "56-65": 3, "65+": 4}).astype(int)

# --- (c) ONE-HOT encoding with drop_first ---
encoded = pd.get_dummies(encoded,
                         columns=["Education_Level", "Marital_Status", "Income_Category"],
                         drop_first=True,
                         dtype=int)

print("Shape after encoding:", encoded.shape)
print("Any non-numeric column left?", list(encoded.select_dtypes(exclude=np.number).columns))
print("Any missing values introduced?", encoded.isnull().sum().sum())
encoded.head(3)

Shape after encoding: (10127, 35)
Any non-numeric column left? []
Any missing values introduced? 0


,Customer_Age,Gender,Dependent_count,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Churn,Avg_Trans_Value,Trans_Per_Month,Contacts_Per_Product,Age_Group,Inactive_x_Contacts,Education_Level_Doctorate,Education_Level_Graduate,Education_Level_High School,Education_Level_Post-Graduate,Education_Level_Uneducated,Education_Level_Unknown,Marital_Status_Married,Marital_Status_Single,Marital_Status_Unknown,Income_Category_$40K - $60K,Income_Category_$60K - $80K,Income_Category_$80K - $120K,Income_Category_Less than $40K,Income_Category_Unknown
0,45,1,3,0,39,5,1,3,12691.0,777,1.335,1144,42,1.625,0.061,0,27.238095,1.076923,0.600000,1,3,0,0,1,0,0,0,1,0,0,0,1,0,0,0
1,49,0,5,0,44,6,1,2,8256.0,864,1.541,1291,33,3.714,0.105,0,39.121212,0.750000,0.333333,2,2,0,1,0,0,0,0,0,1,0,0,0,0,1,0
2,51,1,3,0,36,4,1,0,3418.0,0,2.594,1887,20,2.333,0.000,0,94.350000,0.555556,0.000000,2,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0


**Choice and justification per column:**

| Column | Encoder | Why |
|---|---|---|
| `Gender` | **Label (0/1)** | Only two levels, so one column is enough and no false ordering is possible — one-hot with `drop_first` would produce the identical single column anyway. |
| `Card_Category` | **Ordinal label 0–3** | Blue → Silver → Gold → Platinum is a **real tier ranking**, and the distance between codes carries meaning the model can use. One-hot would add three columns, two of them extremely sparse (Platinum has ~20 customers out of 10,127), which invites overfitting on a handful of rows. |
| `Age_Group` | **Ordinal label 0–4** | I created these bins myself and they are ordered by construction, so preserving the order costs one column instead of four. |
| `Education_Level` | **One-hot, `drop_first=True`** | Education *is* orderable in principle, but 15% of values are `"Unknown"` and there is no honest rank for Unknown — forcing it into a numeric position would assert something false about 1,519 customers. One-hot lets Unknown be its own level. |
| `Marital_Status` | **One-hot, `drop_first=True`** | Purely nominal: Married/Single/Divorced/Unknown have no order at all. Label-encoding them would tell the model that Divorced sits "between" two other statuses, which is meaningless and actively harmful to distance-based methods. |
| `Income_Category` | **One-hot, `drop_first=True`** | Same problem as education: the five brackets are ordered but `"Unknown"` (11%) is not placeable, and the SQL showed churn is **U-shaped** across income rather than increasing — a single ordinal column can only express a straight-line effect, so dummies fit the actual pattern better. |

**Why `drop_first=True`:** with all *k* dummies present, they sum to 1 in every row, which is perfectly
collinear with the intercept (the "dummy variable trap"). That makes logistic-regression coefficients
unstable and inflates the feature count for no information gain — *k−1* columns carry exactly the same
information, with the dropped level acting as the reference category. I use it for the nominal
one-hot columns; I do **not** need it for `Gender`, which is already a single 0/1 column.

## 2.3 Scaling

Separate the features from the target first, then scale the features only — the target must stay as
clean 0/1 labels.

In [ ]:
X_unscaled = encoded.drop(columns="Churn")
y = encoded["Churn"]

print("Before scaling — ranges are wildly different:")
print(X_unscaled[["Credit_Limit", "Avg_Utilization_Ratio", "Total_Trans_Amt", "Trans_Per_Month"]]
      .agg(["min", "max", "std"]).round(2))

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_unscaled), columns=X_unscaled.columns, index=X_unscaled.index)

print("\nAfter StandardScaler — every feature is mean 0, std 1:")
print(X_scaled.describe().T[["mean", "std", "min", "max"]].round(3).head(10))

Before scaling — ranges are wildly different:
     Credit_Limit  Avg_Utilization_Ratio  Total_Trans_Amt  Trans_Per_Month
min       1438.30                   0.00           510.00             0.19
max      34516.00                   1.00         18484.00             9.77
std       9088.78                   0.28          3397.13             0.91

After StandardScaler — every feature is mean 0, std 1:
                          mean  std    min    max
Customer_Age               0.0  1.0 -2.536  3.327
Gender                     0.0  1.0 -0.943  1.060
Dependent_count           -0.0  1.0 -1.806  2.043
Card_Category             -0.0  1.0 -0.251  8.738
Months_on_book            -0.0  1.0 -2.871  2.513
Total_Relationship_Count  -0.0  1.0 -1.810  1.407
Months_Inactive_12_mon     0.0  1.0 -2.317  3.621
Contacts_Count_12_mon     -0.0  1.0 -2.220  3.204
Credit_Limit               0.0  1.0 -0.792  2.848
Total_Revolving_Bal       -0.0  1.0 -1.427  1.662


**Why `StandardScaler` and not `MinMaxScaler`:**

- **The scale problem is real.** `Credit_Limit` runs to 34,516 while `Avg_Utilization_Ratio` runs 0–1.
  K-Means and PCA in Part 3 both work on Euclidean distance, so without scaling the credit limit would
  dominate every distance and the clusters would be nothing but "high limit vs low limit".
- **`MinMaxScaler` is fragile with the outliers this dataset has.** It squeezes everything into [0, 1]
  using the min and max, so the long right tails in `Credit_Limit` and `Total_Trans_Amt` would compress
  the bulk of customers into a narrow band near zero, and a single extreme customer could set the
  scale for everyone.
- **`StandardScaler` centres on the mean and divides by the standard deviation**, which is driven by the
  whole distribution rather than by two extreme points. It keeps the relative spacing between typical
  customers interpretable in standard deviations, and it is the scaler PCA expects, since PCA
  decomposes a covariance/correlation structure that assumes centred data.
- The 0/1 dummy columns are standardised too. That is intentional: leaving them unscaled while scaling
  the continuous columns would silently give them a different weight in the distance calculation.

I would only prefer `MinMaxScaler` if a method required strictly non-negative or bounded inputs (e.g.
some neural nets, or multinomial Naive Bayes) — which nothing in Parts 3–5 does.

## 2.4 Save the modelling dataset

Both the scaled and unscaled versions are kept: **scaled** (`X_scaled`) for K-Means, PCA, KNN and logistic
regression; **unscaled** (`X_unscaled`) for tree-based models, which are indifferent to scale and easier to
interpret in the original units.

They are also written to CSV files and to the database as a checkpoint. The later parts of this notebook
simply keep using the variables in memory.

In [ ]:
X_unscaled.to_csv("X_unscaled.csv", index=False)
X_scaled.to_csv("X_scaled.csv", index=False)
y.to_csv("y.csv", index=False)

# also park the modelling table back in the database
encoded.to_sql("customers_model_ready", conn, if_exists="replace", index=False)

print("Features:", X_unscaled.shape, "| Target:", y.shape)
print("Churn rate:", round(y.mean(), 4))
print("Ready for Part 3 (Clustering)")

Features: (10127, 34) | Target: (10127,)
Churn rate: 0.1607
Ready for Part 3 (Clustering)


### Parts 1–2 summary

- 10,127 customers loaded from Kaggle and stored in SQLite; four SQL queries established the 16.07%
  churn rate, the U-shaped income effect, the strong product-count effect (25.6% churn at 1 product vs
  10.5% at 6) and the behavioural profile of leavers.
- EDA found no true nulls but `"Unknown"` placeholders in three columns, heavy right-skew in the credit
  and spend features, and a bimodal `Total_Trans_Ct` that suggests natural segments.
- Cleaning removed 2 leakage columns, 1 ID column and 1 perfectly collinear column; **no rows dropped**
  and **no values imputed**.
- 5 engineered features (3 ratios, 1 binning, 1 interaction), all encoded per-column with the choice
  justified, then standardised — leaving **34 features** ready for clustering and classification.

---
# Part 3 — Clustering Analysis

## 3.1 Which features should feed K-Means?

K-Means groups rows by **Euclidean distance**, which is designed for continuous columns. My scaled table
contains 14 one-hot dummy columns (education, marital status, income). After `StandardScaler`, a rare
dummy becomes a huge jump (e.g. +3) between "in this category" and "not", so these columns can dominate the
distance. Let me test that worry instead of assuming it: cluster on **all 34 features** and see what
the clusters turn out to be.

In [ ]:
km_all = KMeans(n_clusters=6, n_init=10, random_state=42).fit(X_scaled)
check = X_unscaled.copy()
check["cluster"] = km_all.labels_

income_cols = [c for c in X_unscaled.columns if c.startswith("Income_Category_")]
print("Share of each cluster falling in each income bracket (clusters from ALL 34 features):")
check.groupby("cluster")[income_cols].mean().round(2)

Share of each cluster falling in each income bracket (clusters from ALL 34 features):


,Income_Category_$40K - $60K,Income_Category_$60K - $80K,Income_Category_$80K - $120K,Income_Category_Less than $40K,Income_Category_Unknown
cluster,,,,,
0,1.00,0.00,0.00,0.00,0.00
1,0.00,0.00,0.00,0.99,0.00
2,0.17,0.17,0.19,0.27,0.07
3,0.00,1.00,0.00,0.00,0.00
4,0.00,0.00,0.70,0.00,0.00
5,0.00,0.00,0.00,0.00,1.00


**What I learned:** four of the six clusters are essentially **one income bracket**: one is ~99% "Less than
40K", another 100% "40K-60K", another 100% "Unknown", another 100% "60K-80K". The algorithm has simply
re-discovered the income column instead of finding customer *behaviour*. That is a segmentation the bank
already has and learns nothing new from.

**Decision:** cluster only on the **continuous / ordinal customer-behaviour columns** and leave the
one-hot dummies out. I also drop `Age_Group`, because it is just a binned copy of `Customer_Age` and would
count age twice. Result: 19 features.

In [ ]:
dummy_prefix = ("Education_Level_", "Marital_Status_", "Income_Category_")
cluster_cols = [c for c in X_scaled.columns
                if not c.startswith(dummy_prefix) and c != "Age_Group"]

X_cluster = X_scaled[cluster_cols]
print(len(cluster_cols), "features used for clustering:")
print(cluster_cols)

19 features used for clustering:
['Customer_Age', 'Gender', 'Dependent_count', 'Card_Category', 'Months_on_book', 'Total_Relationship_Count', 'Months_Inactive_12_mon', 'Contacts_Count_12_mon', 'Credit_Limit', 'Total_Revolving_Bal', 'Total_Amt_Chng_Q4_Q1', 'Total_Trans_Amt', 'Total_Trans_Ct', 'Total_Ct_Chng_Q4_Q1', 'Avg_Utilization_Ratio', 'Avg_Trans_Value', 'Trans_Per_Month', 'Contacts_Per_Product', 'Inactive_x_Contacts']


## 3.2 K-Means for K = 2 to 10: elbow curve and silhouette scores

- **Inertia (elbow):** total squared distance from points to their cluster centre. It always falls as K
  grows, so we look for the K where the drop stops being worthwhile.
- **Silhouette score:** for each customer, how much closer they are to their own cluster than to the
  nearest other one. Ranges from -1 to 1; higher is better, and below ~0.25 means weak structure.

In [ ]:
results = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_cluster)
    results.append({"K": k,
                    "inertia": km.inertia_,
                    "silhouette": silhouette_score(X_cluster, km.labels_)})

results = pd.DataFrame(results)
results["inertia_drop"] = -results["inertia"].diff()     # how much inertia fell vs K-1
print(results.round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(results["K"], results["inertia"], marker="o")
axes[0].set_title("Elbow curve"); axes[0].set_xlabel("K"); axes[0].set_ylabel("Inertia")
axes[1].plot(results["K"], results["silhouette"], marker="o", color="darkorange")
axes[1].set_title("Silhouette score"); axes[1].set_xlabel("K"); axes[1].set_ylabel("Silhouette")
plt.tight_layout()
plt.show()

 K    inertia  silhouette  inertia_drop
 2 167857.256       0.234           NaN
 3 153457.892       0.109     14399.364
 4 144691.896       0.098      8765.996
 5 137694.598       0.102      6997.298
 6 131183.226       0.099      6511.372
 7 126496.733       0.098      4686.493
 8 122192.920       0.095      4303.813
 9 118061.796       0.098      4131.124
10 114757.549       0.096      3304.247


<Figure size 1300x450 with 2 Axes>

## 3.3 Choosing the final K

**What the two signals say**

- **Elbow:** there is no sharp elbow — the curve bends softly. The inertia drop is large from K=2 to K=3
  (about 14,400), smaller from 3 to 4 (about 8,800), and then keeps shrinking slowly (about 7,000,
  6,500, 4,700 ... 3,300). The flattening starts around **K = 3 to 4**.
- **Silhouette:** K = 2 scores highest (about 0.23). Beyond that, K = 3 is the best value (about 0.11), and
  K = 4 to 10 all sit between 0.095 and 0.102 with no further gain.

**My choice: K = 3.**

1. K = 2 has the best silhouette, but it just splits off one small premium group (about 14% of customers)
   from everyone else — too coarse to act on.
2. K = 3 has the best silhouette **after** K = 2 and is where the elbow begins to flatten, so both signals
   agree it is the last K that is still "paying for itself". Going to 4+ only shaves off small amounts
   of inertia without improving separation.
3. K = 3 splits the large majority group into two behaviourally different halves that K = 2 hides, which
   is worth having.

**Honest caveat:** silhouette values around 0.1 mean the customers form a **continuum**, not tight islands.
These clusters are useful segments for planning, not natural species.

In [ ]:
K_FINAL = 3
km_final = KMeans(n_clusters=K_FINAL, n_init=10, random_state=42).fit(X_cluster)

# K-Means numbers its clusters randomly, so re-number them by average credit limit (0 = lowest)
# to keep the numbering stable every time the notebook is run.
profile_df = X_unscaled.copy()
profile_df["raw"] = km_final.labels_
order = profile_df.groupby("raw")["Credit_Limit"].mean().sort_values().index
remap = {old: new for new, old in enumerate(order)}
profile_df["Cluster"] = profile_df["raw"].map(remap)

print(profile_df["Cluster"].value_counts().sort_index())

Cluster
0    4900
1    3887
2    1340
Name: count, dtype: int64


## 3.4 Profiling each cluster

Group means in the **original units** (from `X_unscaled`), so the numbers mean something to a human.

In [ ]:
profile = profile_df.groupby("Cluster").agg(
    customers         = ("Gender", "size"),
    share_male        = ("Gender", "mean"),
    age               = ("Customer_Age", "mean"),
    months_on_book    = ("Months_on_book", "mean"),
    products_held     = ("Total_Relationship_Count", "mean"),
    credit_limit      = ("Credit_Limit", "mean"),
    transactions      = ("Total_Trans_Ct", "mean"),
    total_spend       = ("Total_Trans_Amt", "mean"),
    utilisation       = ("Avg_Utilization_Ratio", "mean"),
    months_inactive   = ("Months_Inactive_12_mon", "mean"),
    service_contacts  = ("Contacts_Count_12_mon", "mean"),
    card_tier_0to3    = ("Card_Category", "mean"),
).round(2)

profile.T

Cluster,0,1,2
customers,4900.00,3887.00,1340.00
share_male,0.34,0.57,0.64
age,44.46,49.05,45.26
months_on_book,33.90,38.85,34.89
products_held,4.17,3.92,2.19
credit_limit,5658.88,10025.00,15462.75
transactions,66.58,51.08,98.53
total_spend,3733.23,2721.83,11737.02
utilisation,0.39,0.17,0.16
months_inactive,2.08,2.71,2.23


**Reading the table and naming the clusters** (check that the numbers below match your table; if your
cluster numbers differ, keep the names but move them to the right cluster):

| Cluster | Name | Evidence |
|---|---|---|
| **0** | **Engaged everyday users** | Lowest credit limit (~\$5.7K) but **uses it the most**: highest utilisation (~0.39), ~67 transactions a year, 4+ products, and the **fewest** inactive months and service contacts (~2 each). The card is part of their daily routine. |
| **1** | **Quiet veterans who keep calling** | Oldest (~49) and longest-tenured (~39 months), with the **fewest transactions** (~51), the lowest spend, low utilisation (~0.17), the **most inactive months (~2.7) and the most service contacts (~3.1)**. Long relationship, low usage, lots of friction. |
| **2** | **High-spend premium card users** | Highest credit limit (~\$15K), **~98 transactions and ~\$11.7K annual spend**, upgraded card tier, but only **~2.2 products**. Small group (~13% of customers) who spend heavily on the card but are not tied to other bank products. |

I named them from *behaviour* (usage, tenure, contact) rather than from demographics, because the
demographics are similar across the three (average age differs by under 5 years and tenure by about 5 months).

## 3.5 Visualising the clusters in two dimensions with PCA

The 19 clustering features cannot be plotted, so PCA compresses them into 2 new axes that keep as much of
the variation as possible.

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_cluster)
print("Variance explained by PC1 and PC2:", pca.explained_variance_ratio_.round(3),
      "| total:", round(pca.explained_variance_ratio_.sum(), 3))

# names taken from the profile table in section 3.4
cluster_names = {0: "Engaged everyday users",
                 1: "Quiet veterans who keep calling",
                 2: "High-spend premium card users"}
legend_names = {i: f"{i} - {n}" for i, n in cluster_names.items()}

plt.figure(figsize=(9, 6))
sns.scatterplot(x=coords[:, 0], y=coords[:, 1],
                hue=profile_df["Cluster"].map(legend_names),
                hue_order=list(legend_names.values()),
                palette="Set2", s=12, alpha=0.6)
plt.title("K-Means clusters projected onto the first two principal components")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.legend(title="Cluster", loc="best")
plt.show()

# which original features drive each axis?
loadings = pd.DataFrame(pca.components_.T, index=cluster_cols, columns=["PC1", "PC2"])
loadings.reindex(loadings["PC1"].abs().sort_values(ascending=False).index).head(6).round(2)

Variance explained by PC1 and PC2: [0.188 0.129] | total: 0.317


<Figure size 900x600 with 1 Axes>

,PC1,PC2
Total_Trans_Amt,0.48,0.05
Total_Trans_Ct,0.43,-0.05
Trans_Per_Month,0.43,-0.13
Avg_Trans_Value,0.42,0.09
Total_Relationship_Count,-0.24,-0.22
Months_on_book,-0.17,0.16


**What I learned:** PC1 and PC2 together capture only about a third of the variation (see the printed
number), so some overlap in the picture is expected and does not mean the clustering failed.

- **PC1 is "how much the card is used".** Its top loadings are `Total_Trans_Amt`, `Total_Trans_Ct`,
  `Trans_Per_Month` and `Avg_Trans_Value`. The **premium spenders (cluster 2)** sit far to the right
  (average PC1 about +3.8, against about -0.1 and -1.2 for the others), so they are clearly separated.
- **PC2 is "service friction".** Its top loadings are `Inactive_x_Contacts`, `Contacts_Per_Product` and
  `Contacts_Count_12_mon`. The **quiet veterans (cluster 1)** sit high on it (about +1.2) and the **engaged
  everyday users (cluster 0)** sit low (about -1.2).

Clusters 0 and 1 are separated mainly on PC2 and overlap somewhat at the edges, which matches the modest
silhouette score.

## 3.6 Keep the cluster labels for Part 6

Each customer's cluster number is stored next to their churn flag so Part 6 can compare the two. Churn was
**never** used to build the clusters.

In [ ]:
labels = pd.DataFrame({"Cluster": profile_df["Cluster"], "Churn": y})
print("Customers with a cluster label:", len(labels))
labels.head()

Customers with a cluster label: 10127


,Cluster,Churn
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0


---
# Part 4 — Classification

## 4.1 Setting up the task

`stratify=y` makes the train and test sets keep the same 16% churn rate. Without it, a random split could
give the test set noticeably fewer churners and make the metrics unstable.

**I use `X_unscaled` here, not `X_scaled`.** In Part 2 the scaler was fitted on all rows before any split,
which lets information about the test rows leak into the training features. Correct practice is to fit the
scaler on the **training set only**. I do that below by putting `StandardScaler` *inside* a pipeline for the
model that needs it (logistic regression). Tree models do not care about scale.

**Cross-validation:** the assignment asks for stratified k-fold only when there are fewer than 2,000 rows.
This dataset has 10,127, so one stratified 80/20 split is enough (the test set holds 2,026 customers,
of which about 325 are churners).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_unscaled, y, test_size=0.2, stratify=y, random_state=42)

print("Train:", X_train.shape, "| churn rate:", round(y_train.mean(), 4))
print("Test: ", X_test.shape,  "| churn rate:", round(y_test.mean(), 4))

Train: (8101, 34) | churn rate: 0.1607
Test:  (2026, 34) | churn rate: 0.1604


## 4.2 The three classifiers and why

| Model | Why I chose it |
|---|---|
| **Logistic Regression** | The standard **simple, interpretable baseline**. It draws one straight boundary and its coefficients can be explained to a manager. If a complex model cannot beat it clearly, it is not worth using. It needs scaled inputs, so it goes in a pipeline with `StandardScaler`. |
| **Random Forest** | Many decision trees voting together. It captures **non-linear effects and interactions** without any manual work (for example "low transactions *and* high contacts"), copes with skewed features and needs no scaling. |
| **Gradient Boosting** | Builds trees one after another, each correcting the previous one's mistakes. It usually gives the **best accuracy on tabular data** and is the natural challenger to Random Forest, since it attacks the problem differently (boosting vs bagging). |

I also include a **"dummy" model that predicts "nobody churns" for everyone** — not as a contender, but to
show what accuracy a model earns without learning anything.

## 4.3 Training and evaluating

In [ ]:
models = {
    "Baseline (predict nobody churns)": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "Random Forest":       RandomForestClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(random_state=42),
}

rows, preds, probs = [], {}, {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]        # probability of churn
    preds[name], probs[name] = pred, prob
    rows.append({"Model": name,
                 "Accuracy":  accuracy_score(y_test, pred),
                 "Precision": precision_score(y_test, pred, zero_division=0),
                 "Recall":    recall_score(y_test, pred),
                 "F1":        f1_score(y_test, pred),
                 "ROC AUC":   roc_auc_score(y_test, prob)})

comparison = pd.DataFrame(rows).set_index("Model").round(3)
comparison

,Accuracy,Precision,Recall,F1,ROC AUC
Model,,,,,
Baseline (predict nobody churns),0.840,0.000,0.000,0.000,0.500
Logistic Regression,0.915,0.826,0.597,0.693,0.932
Random Forest,0.957,0.951,0.772,0.852,0.987
Gradient Boosting,0.962,0.956,0.800,0.871,0.989


### Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, name in zip(axes, ["Logistic Regression", "Random Forest", "Gradient Boosting"]):
    ConfusionMatrixDisplay.from_predictions(
        y_test, preds[name], display_labels=["Stayed", "Churned"],
        cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(name)
plt.tight_layout()
plt.show()

<Figure size 1500x420 with 3 Axes>

### ROC curves

In [ ]:
plt.figure(figsize=(6.5, 5))
for name in ["Logistic Regression", "Random Forest", "Gradient Boosting"]:
    RocCurveDisplay.from_predictions(y_test, probs[name], name=name, ax=plt.gca())
plt.plot([0, 1], [0, 1], "k--", label="Random guessing")
plt.title("ROC curves (test set)")
plt.legend()
plt.show()

<Figure size 650x500 with 1 Axes>

### What the numbers say (values from my run; yours should match to within rounding)

| Model | Accuracy | Precision | Recall | F1 | ROC AUC |
|---|---|---|---|---|---|
| Baseline | 0.840 | 0.000 | 0.000 | 0.000 | 0.500 |
| Logistic Regression | 0.915 | 0.826 | 0.597 | 0.693 | 0.932 |
| Random Forest | 0.957 | 0.951 | 0.772 | 0.852 | 0.987 |
| Gradient Boosting | 0.962 | 0.956 | 0.800 | 0.871 | 0.989 |

1. **Accuracy is a trap here.** The do-nothing baseline scores **84% accuracy** while catching **zero**
   churners (recall 0). A model at 91.5% sounds great but is only 7.5 points better than doing nothing.
   For this project the meaningful metrics are **recall and F1 on the churn class**.
2. **Gradient Boosting is the best model on every metric**, followed closely by Random Forest, with Logistic
   Regression clearly behind. The gap between linear and tree models shows churn depends on
   **non-linear patterns and interactions** (like the low-transactions-plus-high-contacts combination),
   which a straight-line boundary cannot represent.
3. **Precision is much higher than recall for all models.** Reading the confusion matrix for Gradient
   Boosting: of ~325 real churners the model catches about 260 and **misses about 65**, while
   raising only about 12 false alarms. It is cautious: when it says "will churn", it is nearly always right,
   but it lets about one churner in five slip through.
4. **Why that matters:** for a bank, **missing a churner (false negative) is usually more expensive than a
   wasted retention call (false positive)**, since losing a customer forfeits all future revenue, while a
   call costs very little. So the models are currently tuned the wrong way round for the business. The
   default 0.5 probability cut-off is the cause, and fixing it (class weights, threshold tuning, SMOTE)
   is exactly the job of **Part 5**.
5. **ROC AUC is high for every real model (0.93 to 0.99)**: the models *rank* customers very well; the
   weakness is only in where the yes/no line is drawn. That again points to threshold tuning as the
   cheap fix.

### Which features drive the predictions?

In [ ]:
best = models["Gradient Boosting"]
importance = pd.Series(best.feature_importances_, index=X_train.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
importance.head(10).sort_values().plot(kind="barh", color="teal")
plt.title("Top 10 features - Gradient Boosting")
plt.xlabel("Importance")
plt.show()

importance.head(10).round(3)

<Figure size 800x500 with 1 Axes>

,0
Total_Trans_Ct,0.302
Total_Revolving_Bal,0.188
Total_Trans_Amt,0.174
Total_Ct_Chng_Q4_Q1,0.094
Total_Relationship_Count,0.082
Avg_Trans_Value,0.043
Total_Amt_Chng_Q4_Q1,0.039
Contacts_Per_Product,0.017
Trans_Per_Month,0.015
Contacts_Count_12_mon,0.011


**What I learned:** the model relies mainly on **card usage and balance**. `Total_Trans_Ct` alone carries
about 30% of the importance, followed by `Total_Revolving_Bal` (~19%) and `Total_Trans_Amt` (~17%), then the
Q4-to-Q1 change in transaction count and `Total_Relationship_Count` (products held, which matches the strong
product-count effect I found with SQL in Part 1). This agrees with the EDA: customers churn when they
**disengage from the card**, not because of who they are. `Customer_Age`, `Gender`, `Card_Category`,
`Dependent_count`, marital status and every education and income dummy each contribute **under 1%**.

Three of my five engineered features (`Avg_Trans_Value`, `Contacts_Per_Product`, `Trans_Per_Month`) reach the
top ten, but with small weights (about 1.5% to 4% each). My honest verdict is that the feature engineering
added a **modest** amount of signal: the raw transaction and balance columns carry most of the
predictive power.

### Parts 3-4 summary

- **Clustering:** including one-hot dummies made K-Means rediscover income brackets, so I clustered on the
  19 behavioural columns only. Elbow and silhouette both point to a small K; I chose **K = 3** (the best
  silhouette after K = 2 and the start of the elbow's flattening), giving three named segments: engaged
  everyday users, quiet veterans who keep calling, and high-spend premium card users. Silhouette ~0.11 means
  the segments overlap.
- **Classification:** Logistic Regression, Random Forest and Gradient Boosting compared against a do-nothing
  baseline. Gradient Boosting wins (F1 ~0.87, ROC AUC ~0.99). Every model has high precision but lower
  recall, so churners are being missed. **Part 5** addresses that.

---
# Part 5 — Imbalance Handling

## 5.1 Setup and the business cost

**Is the data imbalanced enough to need this?** Yes. Only **16.07%** of customers churn, well outside the
40-60% "roughly balanced" band, so the techniques should make a real difference (and the numbers below show
they do).

**Which model?** Gradient Boosting, the winner of Part 4. Every technique below is applied to that same
model so the comparison is fair. I also reuse the **same train/test split** as Part 4 (`X_train`, `X_test`,
`y_train`, `y_test`), so the baseline below matches Part 4 exactly.

**The business cost.** In Part 4 the model was cautious: high precision (0.96) but only 0.80 recall, so it
missed 65 of 325 churners. Whether that is acceptable depends on what each mistake costs. I make this
**assumption explicit** (the bank would replace these with its own figures):

| Mistake | What happens | Assumed cost |
|---|---|---|
| **Missed churner** (false negative) | Customer leaves; we lose their future revenue | **\$300** (about one year of revenue from an average card customer) |
| **False alarm** (false positive) | We spend a retention call and a small incentive on someone who was staying anyway | **\$30** |

So a missed churner is **10 times** as costly as a false alarm. Total cost = 300 x FN + 30 x FP.

In [ ]:
print("Train:", len(y_train), "customers | Test:", len(y_test), "customers,",
      int(y_test.sum()), "churners")

Train: 8101 customers | Test: 2026 customers, 325 churners


In [ ]:
COST_MISS  = 300    # cost of one missed churner (false negative)
COST_ALARM = 30     # cost of one false alarm (false positive)

def evaluate(name, y_true, prob, threshold=0.5):
    """Turn probabilities into yes/no at a threshold and collect every metric."""
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {"Technique": name,
            "Accuracy":  accuracy_score(y_true, pred),
            "Precision": precision_score(y_true, pred),
            "Recall":    recall_score(y_true, pred),
            "F1":        f1_score(y_true, pred),
            "ROC AUC":   roc_auc_score(y_true, prob),
            "Missed (FN)": fn,
            "False alarms (FP)": fp,
            "Total cost ($)": fn * COST_MISS + fp * COST_ALARM}

results = []

## 5.2 Baseline: Gradient Boosting at the default 0.5 cut-off

In [ ]:
baseline = GradientBoostingClassifier(random_state=42).fit(X_train, y_train)
prob_base = baseline.predict_proba(X_test)[:, 1]

results.append(evaluate("Baseline (default 0.5)", y_test, prob_base))
pd.DataFrame(results).round(3)

,Technique,Accuracy,Precision,Recall,F1,ROC AUC,Missed (FN),False alarms (FP),Total cost ($)
0,Baseline (default 0.5),0.962,0.956,0.8,0.871,0.989,65,12,19860


This matches Part 4 (recall 0.80, F1 0.87). The cost column is new: **65 missed churners x \$300 =
\$19,500** of the \$19,860 total. The 12 false alarms cost only \$360. Nearly all of the cost comes from
the churners we fail to catch.

## 5.3 Technique 1 — Class weights

**Idea:** tell the model that a mistake on a churner counts more. With `"balanced"` weights each churner
counts about **3.1x** and each stayer about **0.6x**, so the model stops ignoring the minority class.
Gradient Boosting has no `class_weight` argument, so I pass the same thing through `sample_weight`.

In [ ]:
weights = compute_sample_weight("balanced", y_train)
print("Weight per class:", np.unique(weights).round(2), "(stayer, churner)")

model_cw = GradientBoostingClassifier(random_state=42)
model_cw.fit(X_train, y_train, sample_weight=weights)
prob_cw = model_cw.predict_proba(X_test)[:, 1]

results.append(evaluate("Class weights (0.5)", y_test, prob_cw))
pd.DataFrame(results).round(3)

Weight per class: [0.6  3.11] (stayer, churner)


,Technique,Accuracy,Precision,Recall,F1,ROC AUC,Missed (FN),False alarms (FP),Total cost ($)
0,Baseline (default 0.5),0.962,0.956,0.800,0.871,0.989,65,12,19860
1,Class weights (0.5),0.955,0.812,0.932,0.868,0.991,22,70,8700


## 5.4 Technique 2 — SMOTE

**Idea:** create new **synthetic churners** by interpolating between existing ones until the classes are
balanced. Two rules matter:

1. **Only on the training data.** If SMOTE were applied before the split, synthetic copies of test
   customers would leak into training. Putting SMOTE inside an `imblearn` pipeline guarantees it only ever
   touches the training rows.
2. The test set stays untouched and keeps its real 16% churn rate.

In [ ]:
smote_model = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("gb", GradientBoostingClassifier(random_state=42)),
])
smote_model.fit(X_train, y_train)
prob_smote = smote_model.predict_proba(X_test)[:, 1]

results.append(evaluate("SMOTE (0.5)", y_test, prob_smote))
pd.DataFrame(results).round(3)

,Technique,Accuracy,Precision,Recall,F1,ROC AUC,Missed (FN),False alarms (FP),Total cost ($)
0,Baseline (default 0.5),0.962,0.956,0.800,0.871,0.989,65,12,19860
1,Class weights (0.5),0.955,0.812,0.932,0.868,0.991,22,70,8700
2,SMOTE (0.5),0.960,0.886,0.862,0.874,0.987,45,36,14580


## 5.5 Technique 3 — Threshold tuning

**Idea:** the model outputs a churn *probability*. The default rule "flag if probability >= 0.5" is
arbitrary. Because a missed churner costs 10x a false alarm, we should flag customers **earlier**, at a lower
probability. No retraining is needed.

**Important:** I must not choose the threshold by looking at the test set, or the test result would be
optimistic. Instead I get **out-of-fold probabilities on the training set** (5-fold cross-validation: every
training customer is scored by a model that never saw them), pick the threshold there, and only then apply it
to the test set once.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
prob_oof = cross_val_predict(GradientBoostingClassifier(random_state=42),
                             X_train, y_train, cv=cv, method="predict_proba")[:, 1]

thresholds = np.arange(0.05, 0.95, 0.01)

def total_cost(prob, y_true, t, cost_miss=COST_MISS, cost_alarm=COST_ALARM):
    pred = (prob >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return fn * cost_miss + fp * cost_alarm

costs = np.array([total_cost(prob_oof, y_train, t) for t in thresholds])
best_t = round(float(thresholds[costs.argmin()]), 2)
print("Cheapest threshold on the training folds:", best_t)

Cheapest threshold on the training folds: 0.12


### Precision-recall curve and the cost curve

In [ ]:
precision, recall, pr_thr = precision_recall_curve(y_train, prob_oof)

def pr_at(t):
    pred = (prob_oof >= t).astype(int)
    return recall_score(y_train, pred), precision_score(y_train, pred)

r_def, p_def = pr_at(0.5)
r_best, p_best = pr_at(best_t)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axes[0].plot(recall, precision, color="steelblue")
axes[0].scatter(r_def, p_def, color="gray", s=80, zorder=3, label="default 0.5")
axes[0].scatter(r_best, p_best, color="crimson", s=80, zorder=3, label=f"chosen {best_t}")
axes[0].set_xlabel("Recall"); axes[0].set_ylabel("Precision")
axes[0].set_title("Precision-recall curve (training folds)")
axes[0].legend()

axes[1].plot(thresholds, costs, color="darkorange")
axes[1].axvline(best_t, color="crimson", linestyle="--", label=f"minimum at {best_t}")
axes[1].set_xlabel("Threshold"); axes[1].set_ylabel("Total cost ($)")
axes[1].set_title("Business cost by threshold (training folds)")
axes[1].legend()

plt.tight_layout()
plt.show()

<Figure size 1300x480 with 2 Axes>

**Reading the plots:** moving from the grey dot (0.5) to the red dot (the chosen threshold) lifts recall from
about 0.83 to 0.97 at the price of precision falling from about 0.93 to 0.71. The right-hand plot turns this into money: cost falls
steeply as the threshold drops from 0.5, bottoms out, and then climbs again as false alarms pile up.

The curve is **flat near its minimum**: any threshold from about 0.08 to 0.20 costs within roughly 10% of
the best one (about \$29-31K on the training folds against \$28K at 0.12), whereas the default 0.5 costs
about \$67K. So the exact value of 0.12 matters far less than moving away from 0.5.

**Sanity check from theory.** If the probabilities were perfectly calibrated, the cheapest rule is "flag
whenever P(churn) > cost of a false alarm / (cost of a false alarm + cost of a miss)" = 30 / 330 =
**0.09**. My data-driven threshold is in the same region (0.12), which is reassuring; the gap comes from
imperfect calibration.

In [ ]:
print("How the best threshold moves when the cost assumption changes:")
sens = []
for miss in [150, 300, 600]:
    c = np.array([total_cost(prob_oof, y_train, t, cost_miss=miss) for t in thresholds])
    sens.append({"Cost of a miss": f"${miss}",
                 "Cost ratio (miss : alarm)": f"{miss // COST_ALARM} : 1",
                 "Best threshold": round(float(thresholds[c.argmin()]), 2)})
pd.DataFrame(sens)

How the best threshold moves when the cost assumption changes:


,Cost of a miss,Cost ratio (miss : alarm),Best threshold
0,$150,5 : 1,0.21
1,$300,10 : 1,0.12
2,$600,20 : 1,0.08


**What I learned:** the best threshold is **not a fixed fact** about the data. It moves from about 0.21
(5:1) to 0.12 (10:1) to 0.08 (20:1) as the bank values a saved customer more. That is exactly why the
threshold must be tied to a stated business cost rather than left at 0.5, and why the bank, not the data
scientist, has to supply the real numbers.

In [ ]:
# apply the chosen threshold ONCE to the test set (same baseline model, same probabilities)
results.append(evaluate(f"Threshold tuning ({best_t})", y_test, prob_base, threshold=best_t))

# bonus row: stacking two techniques that do the same job
results.append(evaluate(f"Class weights + threshold {best_t}", y_test, prob_cw, threshold=best_t))

## 5.6 Baseline versus every technique

In [ ]:
table = pd.DataFrame(results).set_index("Technique")
table["Total cost ($)"] = table["Total cost ($)"].astype(int)
table.round(3)

,Accuracy,Precision,Recall,F1,ROC AUC,Missed (FN),False alarms (FP),Total cost ($)
Technique,,,,,,,,
Baseline (default 0.5),0.962,0.956,0.800,0.871,0.989,65,12,19860
Class weights (0.5),0.955,0.812,0.932,0.868,0.991,22,70,8700
SMOTE (0.5),0.960,0.886,0.862,0.874,0.987,45,36,14580
Threshold tuning (0.12),0.930,0.709,0.954,0.814,0.989,15,127,8310
Class weights + threshold 0.12,0.847,0.512,1.000,0.677,0.991,0,310,9300


In [ ]:
dummy = DummyClassifier(strategy="stratified", random_state=42).fit(X_train, y_train)
print("Stratified dummy  F1 / ROC AUC:",
      round(f1_score(y_test, dummy.predict(X_test)), 2), "/",
      round(roc_auc_score(y_test, dummy.predict_proba(X_test)[:, 1]), 2))

final_pred = (prob_base >= best_t).astype(int)
print("Final model       F1 / ROC AUC:",
      round(f1_score(y_test, final_pred), 2), "/",
      round(roc_auc_score(y_test, prob_base), 2))

Stratified dummy  F1 / ROC AUC: 0.17 / 0.51
Final model       F1 / ROC AUC: 0.81 / 0.99


### Reading the table (numbers from my run)

| Technique | Precision | Recall | F1 | ROC AUC | Missed | False alarms | Cost |
|---|---|---|---|---|---|---|---|
| Baseline (0.5) | 0.956 | 0.800 | 0.871 | 0.989 | 65 | 12 | \$19,860 |
| Class weights | 0.812 | 0.932 | 0.868 | 0.991 | 22 | 70 | \$8,700 |
| SMOTE | 0.886 | 0.862 | 0.874 | 0.987 | 45 | 36 | \$14,580 |
| Threshold 0.12 | 0.709 | 0.954 | 0.814 | 0.989 | 15 | 127 | \$8,310 |
| Class weights + threshold | 0.512 | 1.000 | 0.677 | 0.991 | 0 | 310 | \$9,300 |

1. **Every technique trades precision for recall, and the business cost falls.** The baseline's accuracy
   (96.2%) looks best, but it is the most expensive option. Tuned for the cost of mistakes, accuracy drops
   to 93% while total cost **falls by 58%**. This is the whole point of Part 5: the metric that looks best
   is not the one that makes the bank the most money.
2. **Class weights** lift recall from 0.80 to 0.93: 43 more churners caught, for 58 extra false alarms.
   At \$300 versus \$30 that is a good trade.
3. **SMOTE helps least** (cost \$14,580). Its ROC AUC (0.987) is actually slightly *below* the baseline, so it
   did not teach the model anything new, it only moved the boundary a little. A likely reason is that SMOTE
   interpolates between customers, which creates unrealistic synthetic ones for features like the one-hot
   dummies (for example 0.4 of "Married"). Gradient Boosting also already copes well with 16% imbalance.
4. **Threshold tuning is the cheapest to apply and gives the lowest cost** (\$8,310). It needs no retraining
   and leaves ROC AUC unchanged (0.989), because it changes only *where the line is drawn*, not how the
   customers are ranked. It also catches 95% of churners.
5. **Class weights and threshold tuning are effectively tied** (\$8,700 vs \$8,310 is a difference of
   about 1.3 missed churners). With only 325 churners in the test set, that gap is within noise
   and I would not claim one is truly better.
6. **Stacking them backfires.** Both techniques do the same job (push the model toward flagging churners),
   so using both overcorrects: recall hits 100% but there are 310 false alarms and the cost rises again to
   \$9,300. More techniques is not better.

**My choice:** threshold tuning on the baseline model. It is as good as any alternative, needs no retraining,
and the threshold can be re-tuned in minutes when the bank updates its cost figures.

**Limitation:** the costs are my assumptions, and the test set has only 325 churners, so the exact dollar
figures should be read as illustrations of the *direction and size* of the effect.

---
# Part 6 — The Tie-In: Did the Clusters Know About Churn?

In Part 3 the clusters were built **without ever seeing the churn label**, and K was chosen using only the
elbow and silhouette scores. If the clusters nevertheless separate churners from stayers, then the label is
tied to the natural structure of customer behaviour.

## 6.1 Churn rate within each cluster

In [ ]:
overall = labels["Churn"].mean()

by_cluster = labels.groupby("Cluster")["Churn"].agg(customers="size", churners="sum", churn_rate="mean")
by_cluster["lift_vs_overall"]     = by_cluster["churn_rate"] / overall
by_cluster["share_of_customers"]  = by_cluster["customers"] / len(labels)
by_cluster["share_of_churners"]   = by_cluster["churners"] / labels["Churn"].sum()
by_cluster.insert(0, "name", by_cluster.index.map(cluster_names))

print("Overall churn rate:", round(overall, 4))
by_cluster.round(3)

Overall churn rate: 0.1607


,name,customers,churners,churn_rate,lift_vs_overall,share_of_customers,share_of_churners
Cluster,,,,,,,
0,Engaged everyday users,4900,247,0.050,0.314,0.484,0.152
1,Quiet veterans who keep calling,3887,1199,0.308,1.920,0.384,0.737
2,High-spend premium card users,1340,181,0.135,0.841,0.132,0.111


In [ ]:
plt.figure(figsize=(8, 4.5))
rates = by_cluster["churn_rate"] * 100
bars = plt.bar([f"{i} - {cluster_names[i]}" for i in by_cluster.index], rates, color=["#66c2a5", "#e5604d", "#8da0cb"])
plt.axhline(overall * 100, color="black", linestyle="--", label=f"overall {overall:.1%}")
for b, r in zip(bars, rates):
    plt.text(b.get_x() + b.get_width() / 2, r + 0.5, f"{r:.1f}%", ha="center")
plt.ylabel("Churn rate (%)")
plt.title("Churn rate by K-Means cluster")
plt.xticks(rotation=12)
plt.legend()
plt.show()

<Figure size 800x450 with 1 Axes>

## 6.2 How strong is the link?

In [ ]:
table_ct = pd.crosstab(labels["Cluster"], labels["Churn"])
chi2, p_value, dof, _ = chi2_contingency(table_ct)
cramers_v = np.sqrt(chi2 / (table_ct.values.sum() * (min(table_ct.shape) - 1)))
print(f"Chi-square test: chi2 = {chi2:.0f}, p-value = {p_value:.1e}, Cramer's V = {cramers_v:.2f}")

# How well does knowing ONLY the cluster predict churn?
cluster_score = labels["Cluster"].map(by_cluster["churn_rate"])
print("ROC AUC using cluster alone:", round(roc_auc_score(labels["Churn"], cluster_score), 3))

# What if we flagged only the riskiest cluster?
flag = (labels["Cluster"] == 1).astype(int)
print("Flag only cluster 1 -> precision:", round(precision_score(labels["Churn"], flag), 3),
      "| recall:", round(recall_score(labels["Churn"], flag), 3))

Chi-square test: chi2 = 1078, p-value = 8.6e-235, Cramer's V = 0.33
ROC AUC using cluster alone: 0.73
Flag only cluster 1 -> precision: 0.308 | recall: 0.737


## 6.3 What this tells us about the data

**Yes, two of the three clusters are strong predictors.**

- **Cluster 1, "Quiet veterans who keep calling", is the high-risk segment: 30.8% churn, about 1.9x the
  overall rate.** It is 38% of customers but holds **74% of all churners** (1,199 of 1,627).
- **Cluster 0, "Engaged everyday users", is the safest: 5.0% churn**, less than a third of the overall
  rate.
- **Cluster 2, "High-spend premium card users", sits near the middle at 13.5%**, slightly below average. Heavy
  spending does not buy the loyalty that daily use does: these customers hold only ~2.2 products.
- The differences are not chance: the chi-square p-value is essentially zero and Cramer's V is 0.33, a
  moderate association.

**What that says about the data:**

1. **Churn is a behavioural story.** The clustering used only usage, balance, tenure, products and
   service-contact features and never saw the label, yet it separated a 5% group from a 31% group. That agrees
   with the SQL and EDA in Part 1 and with the feature importances in Part 4 (transaction count, revolving
   balance, spend). Churners are the people drifting away from using the card and contacting the bank more.
2. **But clusters are risk tiers, not predictions.** Using cluster membership alone gives ROC AUC of only
   **0.73**, versus **0.99** for Gradient Boosting. Even in the riskiest cluster **69% of customers do not
   churn**: flagging all of cluster 1 would catch 74% of churners but with only 31% precision.
3. **That gap is expected from Part 3.** The silhouette score of about 0.11 showed the customers form a
   continuum rather than separate islands. Churn risk rises smoothly with disengagement, and a supervised
   model can draw the sharp line *inside* each cluster that K-Means cannot.
4. **The clusters add no new information for a model.** They are built from the same features the classifier
   already uses, so they are best treated as a **communication tool** (segments the marketing team can
   understand and target) rather than as an extra predictive feature.

*One technical note:* the clusters were fitted on all customers. That is harmless for this analysis since
no labels were used, but if cluster membership were ever used as a model feature it would have to be fitted
inside each training fold.

---
# Part 7 — Conclusion and Recommendations

## What we found

| Stage | Key finding |
|---|---|
| **Problem** | 1,627 of 10,127 credit-card customers (16.07%) churned. The target is imbalanced about 1:5. |
| **Data quality** | Two columns were another model's predictions (target leakage) and were removed; missing values were hidden as the word "Unknown" in 3 columns and were kept as a category, not guessed. |
| **EDA and SQL** | Churners are **disengaged before they leave**: 44.9 transactions a year vs 68.7 for stayers, and a Q4-to-Q1 change in transaction count of 0.55 vs 0.74. Customers holding 1-2 products churn at 26-28% against about 10-12% for 4-6 products. |
| **Segments** | Three behavioural segments. The **"quiet veterans who keep calling"** hold 74% of all churners at a 30.8% churn rate. The **"engaged everyday users"** churn at only 5.0%. |
| **Prediction** | Gradient Boosting was best (F1 0.87, ROC AUC 0.99). Its predictions rest mainly on transaction count, revolving balance and total spend. Demographics contributed under 1% each. |
| **Imbalance** | At the default cut-off the model missed 20% of churners. Re-tuning the threshold to 0.12 cut missed churners from 65 to 15 and assumed cost by 58%. |

## Recommendations

1. **Deploy Gradient Boosting as a monthly churn-risk score, using the cost-based threshold.** Flag customers
   above the tuned threshold for a retention contact. Re-tune the threshold whenever the bank updates what a
   lost customer and a retention call really cost; it can be done in minutes without retraining.
2. **Focus retention effort on the "quiet veterans" segment.** It is 38% of customers but nearly three-quarters
   of churners. These customers are long-standing (about 39 months), hold about 4 products, but transact
   least and have the most inactive months (about 2.7) and service contacts (about 3.1). The pattern suggests
   **unresolved service friction plus low usage**, so the offer should combine problem resolution with a
   reason to use the card (for example a spend-based reward), not just a discount.
3. **Set up an early-warning trigger on falling card usage.** Transaction count and its Q4-to-Q1 change are among
   the strongest signals. A rule such as "transactions have dropped sharply versus last quarter" can alert the
   bank *before* the customer has decided to leave.
4. **Cross-sell customers who hold only one or two products.** They churn at 26-28%, more than double the rate
   of customers with 4-6 products. This is an association, so the bank should test it with a small controlled
   campaign before assuming that adding a product *causes* loyalty.
5. **Do not target by demographics.** Age, gender, income and education contributed under 1% each to the
   model. Campaigns should be driven by behaviour, which is what predicts leaving.
6. **Protect the premium spenders.** They are only 13% of customers but spend about \$11,700 a year, three to four
   times the other segments, on just 2.2 products. Their churn rate (13.5%) is near average, but each loss
   costs more, so they deserve a dedicated relationship offer.

## Limitations

- **One snapshot, not a time series.** We know who had left by the time the data was taken, but not the order of
  events, so we cannot say how many months of warning the model really gives. A proper test would train on
  earlier months and predict later ones.
- **Association, not causation.** The data shows what churners look like; it does not prove that changing
  their behaviour would keep them.
- **The business costs are assumptions** (\$300 per missed churner, \$30 per false alarm). The dollar figures
  illustrate the size of the effect; the real threshold must come from the bank's own numbers.
- **Small test set for the minority class.** The test set has only 325 churners, so small differences (for
  example class weights versus threshold tuning) are within noise. A single split was used because the dataset
  has over 2,000 rows, but repeating the evaluation across several folds would give firmer estimates.
- **Models were not hyperparameter-tuned.** All used near-default settings. Gains from tuning are likely, but
  the ranking of models is unlikely to change given how large the gap to Logistic Regression was.
- **Clusters are overlapping** (silhouette about 0.11), and one-hot columns had to be excluded from
  clustering. The segment names describe averages, not every customer in them.
- **A small data-handling issue from Part 2:** the scaler was fitted on all rows before splitting. Parts 4-5
  use the unscaled features (scaling for logistic regression happens inside the training pipeline), and the
  effect on Part 3 is negligible because clustering used no labels.

## Next steps

1. Test the retention offers with a **randomised pilot** (offer vs no offer to flagged customers) to measure
   how many churners are truly saved.
2. **Tune and cross-validate** the models, and explain individual predictions with SHAP values.
3. Rebuild the pipeline on **monthly snapshots** so the model is validated on the future, as it would be used.
4. Replace the assumed costs with the bank's **actual customer value and campaign cost**.

In [ ]:
notebook_end_time = time.time()
total_notebook_runtime = notebook_end_time - notebook_start_time
print(f"Notebook execution finished at: {time.ctime(notebook_end_time)}")
print(f"Total notebook runtime: {total_notebook_runtime:.2f} seconds")
print(f"Total notebook runtime: {(total_notebook_runtime)/60:.2f} minutes")

Notebook execution finished at: Fri Oct  2 22:59:14 2026
Total notebook runtime: 109.53 seconds
Total notebook runtime: 1.83 minutes
